In [1]:
import sys
sys.path.append("..")

In [2]:
import numpy as np
import pandas as pd
import os
import plotly.express as px
from sklearn.model_selection import ParameterGrid

from src.data import *
from src.utils import *
from src.model import *
from src.recourse import *

In [3]:
from sklearn.linear_model import LogisticRegression
import plotly.express as px

In [4]:
# Linf
# Grid Search

def getStats(x: np.ndarray, theta: np.ndarray, x0: np.ndarray, lamb):
    return np.log(1 + np.exp(-(x @ theta))) + \
        (lamb * (np.linalg.norm(x0 - x, ord=1)))

def calThetaAdv_l1(xP: np.ndarray, theta0: np.ndarray, alpha):
    # xP has bias
    thetaP = theta0.copy()
    i = np.argmax(np.abs(xP))
    thetaP[i] -= (alpha * np.sign(xP[i]))

    return thetaP

def calThetaAdv_linf(xP: np.ndarray, weights: np.ndarray, bias, alpha):
    # xP does not have bias
    weights_adv = weights - (alpha * np.sign(xP))

    for i in range(len(xP)):
        if np.sign(xP[i]) == 0:
            weights_adv[i] = weights_adv[i] - (alpha * np.sign(weights_adv[i]))
    bias_adv = bias - alpha

    return np.hstack((weights_adv, bias_adv))

def getAllPossibleX(x0: np.ndarray, length=6, step_size=0.1):
    # x0 don't have bias
    x0 = np.hstack((x0, np.array([1])))
    d = [[(length - 1) * -abs(x), length * abs(x)] for x in x0]
    d[x0.size- 1][0] = 1
    d[x0.size- 1][1] = 1

    delta_x = [np.arange(d[i][0], d[i][1] + step_size/2, step_size) for i in range(len(x0))]
    X = np.array(np.meshgrid(*delta_x)).T.reshape(-1, x0.shape[0])
    return np.round(X, decimals=5)

def searchGrid_linf(X, x0_withBias, theta0, bias0, alpha, lamb):
    Js = np.apply_along_axis(lambda x : getStats(x, calThetaAdv_linf(x[:-1], theta0, bias0, alpha), x0_withBias, lamb), arr=X, axis=1)
    Js_min_i = np.argmin(Js)
    xR_new_GS = X[Js_min_i]

    return xR_new_GS[:-1]

In [5]:
# def calThetaAdv_linf(xP: np.ndarray, weights: np.ndarray, bias, alpha):
#     # xP does not have bias
#     weights_adv = weights - (alpha * np.sign(xP))

#     for i in range(len(xP)):
#         if np.sign(xP[i]) == 0:
#             weights_adv[i] = weights_adv[i] - (alpha * np.sign(weights_adv[i]))
#     bias_adv = bias - alpha

#     # return np.concat((weights_adv, bias_adv))
#     return (weights_adv, bias_adv)

In [6]:
def getTheta0AndBias0 (ret):
    try:
        theta0 = ret['theta_0']['out.weight'][0].numpy().astype(np.float64)
        bias0 = ret['theta_0']['out.bias'].numpy().astype(np.float64) 
    except TypeError:
        try: 
            theta0 = ret['theta_0'][0][0][0][0].numpy().astype(np.float64)
            bias0 = ret['theta_0'][0][0][1].numpy().astype(np.float64)
        except IndexError:
            theta0 = ret['theta_0'][0][0][0].numpy().astype(np.float64)
            bias0 = ret['theta_0'][0][1].numpy().astype(np.float64)
    
    return theta0, bias0

### RBR Files

In [14]:
file_path = "../results/cost_validity_latest/lr_synthesis_alg1_lamb0.2_new.pickle"
ret = pd.read_pickle(file_path)

x0 = ret['x_0'][6][0]
x0_withBias = np.hstack((x0, np.array([1])))
xR_old = ret['x_r'][6][0]
xR_old_withBias = np.hstack((xR_old, np.array([1])))

# theta0 = ret['theta_0'][0][0][0][0].numpy().astype(np.float64)
# bias0 = ret['theta_0'][0][0][1].numpy().astype(np.float64) 
theta0, bias0 = getTheta0AndBias0(ret)
# divider = np.linalg.norm(theta0)
# theta0 = ret['theta_0'][0][0][0][0].numpy().astype(np.float64) / divider
# bias0 = ret['theta_0'][0][0][1].numpy().astype(np.float64) /divider
theta0_withBias = np.hstack((theta0, bias0))

alpha = 0.7
lamb = 0.2

In [15]:
lInfR = LARRecourse(weights=theta0, bias=bias0, alpha=alpha , lamb=lamb)
xR_new = lInfR.get_recourse(x0)
xR_new_withBias = np.hstack((xR_new, np.array([1])))

In [24]:
X = getAllPossibleX(x0, length=3, step_size=0.02)

In [25]:
# Js = np.apply_along_axis(lambda x : getStats(x, calThetaAdv_linf(x[:-1], theta0, bias0, alpha), x0_withBias, lamb), arr=X, axis=1)
# Js_min_i = np.argmin(Js)
# xR_new_GS = X[Js_min_i]

xR_new_GS = searchGrid_linf(X, x0_withBias, theta0, bias0, alpha, lamb)

In [28]:
J_xR_old = getStats(xR_old_withBias, calThetaAdv_linf(xR_old, theta0, bias0, alpha), x0_withBias, lamb)
J_xR_new = getStats(xR_new_withBias, calThetaAdv_linf(xR_new, theta0, bias0, alpha), x0_withBias, lamb)
J_xR_new_GS = getStats(np.hstack((xR_new_GS, np.array([1]))), calThetaAdv_linf(xR_new_GS, theta0, bias0, alpha), x0_withBias, lamb)

print(f"X0 : {x0}")
print(f"Theta0 : {theta0_withBias}")
print(f"Alpha: {alpha}")
print(f"Lamb: {lamb}")
print(f"XR Old Linf: {xR_old}, J : {J_xR_old}")
print(f"XR New Linf: {xR_new}, J : {J_xR_new}")
print(f"XR GS: {xR_new_GS}, J : {J_xR_new_GS}")

X0 : [1.8468858  5.56784421]
Theta0 : [ 1.02545357 -0.17008451 -0.95659643]
Alpha: 0.7
Lamb: 0.2
XR Old Linf: [3.6570955 0.       ], J : 2.428894875433304
XR New Linf: [3.6570955 0.       ], J : 2.428894875433304
XR GS: [3.66623 0.00431], J : 2.4303374626388607


In [51]:
# Newly Generated Files

file_path = "../results/cost_validity_latest/lr_synthesis_alg1_lamb0.3_new.pickle"
ret = pd.read_pickle(file_path)

x0 = ret['x_0'][0][98]
x0_withBias = np.hstack((x0, np.array([1])))
xR_old = ret['x_r'][0][98]
xR_old_withBias = np.hstack((xR_old, np.array([1])))

theta0 = ret['theta_0']['out.weight'][0].numpy().astype(np.float64)
bias0 = ret['theta_0']['out.bias'].numpy().astype(np.float64)
# divider = np.linalg.norm(theta0, 2)
# theta0 = theta0 / divider
# bias0 = bias0 /divider
theta0_withBias = np.hstack((theta0, bias0))

alpha = 0.02
lamb = 0.3

tmp = ret['x_r']

In [52]:
lInfR = LARRecourse(weights=theta0, bias=bias0, alpha=alpha , lamb=lamb)
xR_new = lInfR.get_recourse(x0)
xR_new_withBias = np.hstack((xR_new, np.array([1])))

In [58]:
J_xR_old = getStats(xR_old_withBias, calThetaAdv_linf(xR_old, theta0, bias0, alpha), x0_withBias, lamb)
J_xR_new = getStats(xR_new_withBias, calThetaAdv_linf(xR_new, theta0, bias0, alpha), x0_withBias, lamb)

print(f"X0 : {x0}")
print(f"Theta0 : {theta0_withBias}")
print(f"Alpha: {alpha}")
print(f"Lamb: {lamb}")
print(f"XR Old Linf: {xR_old}, J : {J_xR_old}")
print(f"XR New Linf: {xR_new}, J : {J_xR_new}")

X0 : [1.90852233 5.06407279]
Theta0 : [ 0.5587728  -0.30975455  0.46848744]
Alpha: 0.02
Lamb: 0.3
XR Old Linf: [1.90852233 5.06407279], J : 0.7943803330939633
XR New Linf: [1.90852233 5.06407279], J : 0.7943803330939633


In [ ]:
# Newly Generated Files

file_path = "../results/cost_validity_latest/lr_sba_alg1_lamb0.2.pickle"
ret = pd.read_pickle(file_path)

x0 = ret['x_0'][5][5]
x0_withBias = np.hstack((x0, np.array([1])))
xR_old = ret['x_r'][5][5]
xR_old_withBias = np.hstack((xR_old, np.array([1])))

try:
    theta0 = ret['theta_0']['out.weight'][0].numpy().astype(np.float64)
    bias0 = ret['theta_0']['out.bias'].numpy().astype(np.float64) 
except TypeError:
    try: 
        theta0 = ret['theta_0'][0][0][0][0].numpy().astype(np.float64)
        bias0 = ret['theta_0'][0][0][1].numpy().astype(np.float64)
    except IndexError:
        theta0 = ret['theta_0'][0][0][0].numpy().astype(np.float64)
        bias0 = ret['theta_0'][0][1].numpy().astype(np.float64)
# divider = np.linalg.norm(theta0, 2)
# theta0 = theta0 / divider
# bias0 = bias0 /divider
theta0_withBias = np.hstack((theta0, bias0))

alpha = 0.1
lamb = 0.2

tmp = ret['x_r']

lInfR = LARRecourse(weights=theta0, bias=bias0, alpha=alpha , lamb=lamb)
xR_new = lInfR.get_recourse(x0)
xR_new_withBias = np.hstack((xR_new, np.array([1])))

J_xR_old = getStats(xR_old_withBias, calThetaAdv_linf(xR_old, theta0, bias0, alpha), x0_withBias, lamb)
J_xR_new = getStats(xR_new_withBias, calThetaAdv_linf(xR_new, theta0, bias0, alpha), x0_withBias, lamb)

print(f"X0 : {x0}")
print(f"Theta0 : {theta0_withBias}")
print(f"Alpha: {alpha}")
print(f"Lamb: {lamb}")
print(f"XR Old Linf: {xR_old}, J : {J_xR_old}")
print(f"XR New Linf: {xR_new}, J : {J_xR_new}")

X0 : [-0.26558718 -0.06867307  0.49755874 -0.4268808  -0.03345935  1.76401877
  1.62572474 -0.27119077 -0.30590044  0.71882223  1.20519906  0.
  1.          0.        ]
Theta0 : [ 0.77824163 -0.27275857  0.11780936 -0.12219541 -0.05773614  0.21237242
 -0.06366055  0.29351866  0.24285752 -0.55687368 -0.37835842  0.67323387
  0.42163563  0.08462106 -0.14200011]
Alpha: 0.1
Lamb: 0.2
XR Old Linf: [ 0.87201303 -0.06867307  0.49755874 -0.4268808  -0.03345935  1.76401877
  1.62572474 -0.27119077 -0.30590044  0.          1.20519906  0.
  1.          0.        ], J : 1.1455240317853597
XR New Linf: [ 3.08372231 -0.06867307  0.49755874 -0.4268808  -0.03345935  1.76401877
  1.62572474 -0.27119077 -0.30590044  0.71882223  1.20519906  0.
  1.          0.        ], J : 1.0192493983840216


In [8]:
# Checking whether validity is non-monotonic

# alphas = np.linspace(0.02, 1,50)
# xRs = np.empty((len(alphas), len(x0)))

# for i, alpha in enumerate(alphas):

#     tmpL1 = LARRecourse(weights=theta0, bias=bias0, alpha=alpha , lamb=0.2)
#     xRs[i] = tmpL1.get_recourse(x0)

#     clf = LogisticRegression()
#     weightsR, biasR = calThetaAdv_linf(xRs[i], theta0, bias0, alpha)
#     clf.coef_ = weightsR.reshape(1,-1)
#     clf.intercept_ = biasR
#     clf.classes_ = np.array([0,1])
#     print(f"-----------------------------------") 
#     print(f"alpha {alpha}")
#     print(f"x0: {x0}")
#     print(f"XR: {xRs[i]}")
#     print(f"ThetaP: {weightsR}, {biasR}")   
#     print(f"Prob: {clf.predict_proba(xRs[i].reshape(1,-1))[0,1]}")

#     # J = RecourseCost(x_0=x0, lamb=0.4)
#     # print(J.eval(x=xRs[i], weights= weightsR, bias=biasR, breakdown=True))

### Our Files

In [9]:
file_path = "../results/recourse/lr_german_alg1_0.001_0.5_4.pkl"
ret = pd.read_pickle(file_path)
ret

,algorithm,seed,alpha,lambda,i,x_0,x_r,theta_0
0,Alg1,4,0.5,0.001,0,"[2.2482, 2.7256, -0.7516, 0.0, 0.0, 0.0, 1.0]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]","[-0.4105, -0.1379, 0.1719, -0.2133, -0.3401, 0..."
1,Alg1,4,0.5,0.001,1,"[3.2438, 4.3886, -1.2794, 0.0, 0.0, 0.0, 1.0]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]","[-0.4105, -0.1379, 0.1719, -0.2133, -0.3401, 0..."
2,Alg1,4,0.5,0.001,2,"[2.2482, 0.6468, -0.4878, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]","[-0.4105, -0.1379, 0.1719, -0.2133, -0.3401, 0..."
3,Alg1,4,0.5,0.001,3,"[2.2482, 0.3675, -1.0155, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]","[-0.4105, -0.1379, 0.1719, -0.2133, -0.3401, 0..."
4,Alg1,4,0.5,0.001,4,"[2.2482, 1.8487, -1.0155, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]","[-0.4105, -0.1379, 0.1719, -0.2133, -0.3401, 0..."
5,Alg1,4,0.5,0.001,5,"[2.746, 4.4921, 1.9749, 0.0, 0.0, 0.0, 1.0]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]","[-0.4105, -0.1379, 0.1719, -0.2133, -0.3401, 0..."
6,Alg1,4,0.5,0.001,6,"[3.2438, 3.7162, 2.4146, 0.0, 0.0, 0.0, 1.0]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]","[-0.4105, -0.1379, 0.1719, -0.2133, -0.3401, 0..."
7,Alg1,4,0.5,0.001,7,"[1.2526, 2.2333, -0.6637, 1.0, 0.0, 0.0, 0.0]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]","[-0.4105, -0.1379, 0.1719, -0.2133, -0.3401, 0..."
8,Alg1,4,0.5,0.001,8,"[1.2526, 1.5619, 0.3917, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]","[-0.4105, -0.1379, 0.1719, -0.2133, -0.3401, 0..."
9,Alg1,4,0.5,0.001,9,"[1.2526, 2.2311, -0.3998, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]","[-0.4105, -0.1379, 0.1719, -0.2133, -0.3401, 0..."


In [150]:
ind = 0

x0 =  np.hstack((ret['x_0'][ind], [1]))
xr = np.hstack((ret['x_r'][ind], [1]))
theta0 = ret['theta_0'][ind]
alpha = ret['alpha'][ind]
# lamb = ret['lambda'][ind]
lamb = 0.001
l1 = L1Recourse(weights= theta0[:-1], bias= theta0[[-1]], alpha= alpha, lamb= lamb)

thetaP = calThetaAdv_l1(xr, theta0, alpha)
# thetaP = theta0.copy()
# thetaP[11] -= 0.1

x0_pt = torch.from_numpy(x0)
theta0_pt = torch.from_numpy(theta0)
thetaP_pt = torch.from_numpy(thetaP)
A, b = l1.getConstraints(thetaP)

xr_new = l1._runPSDInvScalingMainDs(x0_pt, thetaP_pt, A, b, 
                           abstol=1e-12, n_epochs=7000, lr0=10, step_size=30, gamma=0.95)
# xr_new, loss_tracker, lr_tracker = l1._runPSDInvScalingMainDs(x0_pt, thetaP_pt, A, b, 
#                            abstol=1e-12, n_epochs=7000, lr0=1, step_size=30, gamma=0.95)

j_old = getStats(xr, calThetaAdv_l1(xr, theta0, alpha), x0, lamb)
# j_new = getStats(xr_new, calThetaAdv_l1(xr_new, theta0, alpha), x0, lamb)
j_new = getStats(xr_new, calThetaAdv_l1(xr_new, theta0, alpha), x0, lamb)

0.056034130955285734 - lr0 0.5 | 
0.038888112188353585 - lr0 30 |


In [151]:
print(f"J(old): {j_old}")
print(f"J(new): {j_new}")

J(old): 0.8012942414785201
J(new): 0.038896373187609384


In [152]:
print("cost old: ", np.sum(np.abs(xr - x0)))
print("cost new: ", np.sum(np.abs(xr_new - x0)))

cost old:  0.0
cost new:  30.4114230785393


In [101]:
px.scatter(lr_tracker)

NameError: name 'lr_tracker' is not defined

Do not Touch below

In [87]:
file_path = "../results/recourse/lr_sba_L1PSD_0.001_0.1_4.pkl"
ret = pd.read_pickle(file_path)

In [88]:
num_i = 5
l1 = L1Recourse(weights= ret.loc[num_i, 'theta_0'][:-1], bias= ret.loc[num_i, 'theta_0'][[-1]], alpha= ret.loc[num_i, 'alpha'], lamb= ret.loc[num_i, 'lambda'])
xR = l1.get_recourse(ret.loc[num_i, 'x_0'])

ValueError: setting an array element with a sequence. The requested array would exceed the maximum number of dimension of 1.

In [9]:
# Saved File
xP_old_withB = np.hstack((ret['x_r'][num_i], [1]))
# New
xP_new_withB = np.hstack((xR, [1]))

In [10]:
if "L1" in l1.name:
    thetaP_old = calThetaAdv_l1(xP_old_withB, ret.loc[num_i, 'theta_0'], ret.loc[num_i, 'alpha'])
    thetaP_new = calThetaAdv_l1(xP_new_withB, ret.loc[num_i, 'theta_0'], ret.loc[num_i, 'alpha'])
else:
    thetaP_old = calThetaAdv_linf(xP_old_withB[:-1], ret.loc[num_i, 'theta_0'][:-1], ret.loc[num_i, 'theta_0'][[-1]], ret.loc[num_i, 'alpha'])
    thetaP_new = calThetaAdv_linf(xP_new_withB[:-1], ret.loc[num_i, 'theta_0'][:-1], ret.loc[num_i, 'theta_0'][[-1]], ret.loc[num_i, 'alpha'])


j_old = getStats(xP_old_withB, thetaP_old, np.hstack((ret.loc[num_i, 'x_0'], [1])), ret.loc[num_i, 'lambda'])
j_new = getStats(xP_new_withB, thetaP_new, np.hstack((ret.loc[num_i, 'x_0'], [1])), ret.loc[num_i, 'lambda'])

In [11]:
print(f"{l1.name}")
print(f"xP (old): {xP_old_withB.round(4)}")
print(f"J (old): {j_old}")
print(f"Cost (old): {np.linalg.norm(xP_old_withB - np.hstack((ret.loc[num_i, 'x_0'], [1])), ord=1)}")
print(f"xP (new): {xP_new_withB.round(4)}")
print(f"J (new): {j_new}")
print(f"Cost (new): {np.linalg.norm(xP_new_withB - np.hstack((ret.loc[num_i, 'x_0'], [1])), ord=1)}")

L1PSD
xP (old): [ 6.56000e-01  8.69500e-01  6.83700e-01  8.81300e-01  4.08800e-01
  8.10000e-03 -7.17800e-01 -3.13100e-01 -3.31800e-01  7.66000e-02
  5.38200e-01 -5.47090e+00  5.44700e-01  4.13200e+00  2.06805e+01
  4.07420e+00  3.48270e+00  1.74900e-01  4.08800e-01  6.15300e-01
 -0.00000e+00 -1.82500e-01  5.92700e-01  9.02200e-01 -8.58000e-02
  1.00000e+00]
J (old): 0.021297585363901837
Cost (old): 17.598300000000002
xP (new): [-4.9000e-02  1.0001e+00  8.5360e-01  8.6890e-01 -3.2570e-01 -2.2650e-01
 -3.7670e-01 -3.1310e-01 -2.6530e-01 -1.9280e-01  3.4680e-01 -6.6542e+00
  7.4660e-01  3.7225e+00  2.0994e+01  3.7004e+00  3.2002e+00  2.3280e-01
 -3.2570e-01  7.9000e-03  0.0000e+00  0.0000e+00  1.0000e+00  1.9720e-01
  0.0000e+00  1.0000e+00]
J (new): 0.012004123755505023
Cost (new): 11.565317277496018


In [84]:
file_path = "../results/recourse/lr_sba_Alg1_0.001_0.1_4.pkl"
ret = pd.read_pickle(file_path)

num_i = 36
l1 = LARRecourse(weights= ret.loc[num_i, 'theta_0'][:-1], bias= ret.loc[num_i, 'theta_0'][[-1]], alpha= ret.loc[num_i, 'alpha'], lamb= ret.loc[num_i, 'lambda'])
xR = l1.get_recourse(ret.loc[num_i, 'x_0'])
# Saved File
xP_old_withB = np.hstack((ret['x_r'][num_i], [1]))
# New
xP_new_withB = np.hstack((xR, [1]))

if "L1" in l1.name:
    thetaP_old = calThetaAdv_l1(xP_old_withB, ret.loc[num_i, 'theta_0'], ret.loc[num_i, 'alpha'])
    thetaP_new = calThetaAdv_l1(xP_new_withB, ret.loc[num_i, 'theta_0'], ret.loc[num_i, 'alpha'])
else:
    thetaP_old = calThetaAdv_linf(xP_old_withB[:-1], ret.loc[num_i, 'theta_0'][:-1], ret.loc[num_i, 'theta_0'][[-1]], ret.loc[num_i, 'alpha'])
    thetaP_new = calThetaAdv_linf(xP_new_withB[:-1], ret.loc[num_i, 'theta_0'][:-1], ret.loc[num_i, 'theta_0'][[-1]], ret.loc[num_i, 'alpha'])


j_old = getStats(xP_old_withB, thetaP_old, np.hstack((ret.loc[num_i, 'x_0'], [1])), ret.loc[num_i, 'lambda'])
j_new = getStats(xP_new_withB, thetaP_new, np.hstack((ret.loc[num_i, 'x_0'], [1])), ret.loc[num_i, 'lambda'])

In [85]:
print(f"{l1.name}")
print(f"xP (old): {xP_old_withB.round(4)}")
print(f"J (old): {j_old}")
print(f"Cost (old): {np.linalg.norm(xP_old_withB - np.hstack((ret.loc[num_i, 'x_0'], [1])), ord=1)}")
print(f"xP (new): {xP_new_withB.round(4)}")
print(f"J (new): {j_new}")
print(f"Cost (new): {np.linalg.norm(xP_new_withB - np.hstack((ret.loc[num_i, 'x_0'], [1])), ord=1)}")

alg1
xP (old): [-4.90000e-02  1.00010e+00  8.53500e-01  8.68900e-01 -3.82000e-01
 -2.26500e-01 -3.76700e-01 -3.13100e-01 -2.65300e-01 -1.92800e-01
  3.46800e-01 -8.20070e+00  7.46600e-01  3.72250e+00  2.30563e+01
  3.70050e+00  3.20020e+00  2.32800e-01 -3.82000e-01  7.90000e-03
  0.00000e+00  0.00000e+00  1.00000e+00  0.00000e+00  0.00000e+00
  1.00000e+00]
J (old): 0.0110374758878323
Cost (old): 10.7392
xP (new): [-4.90000e-02  1.00010e+00  8.53500e-01  8.68900e-01 -3.82000e-01
 -2.26500e-01 -3.76700e-01 -3.13100e-01 -2.65300e-01 -1.92800e-01
  3.46800e-01 -8.20040e+00  7.46600e-01  3.72250e+00  2.30563e+01
  3.70050e+00  3.20020e+00  2.32800e-01 -3.82000e-01  7.90000e-03
  0.00000e+00  0.00000e+00  1.00000e+00  0.00000e+00  0.00000e+00
  1.00000e+00]
J (new): 0.011037475695809915
Cost (new): 10.738861264541779


### Index Fix for L1PSD

In [ ]:
# file_path = "../results/recourse/nn_sba_L1PSD_0.7_0.1_0.pkl"
# ret = pd.read_pickle(file_path)

# train_data, test_data = SBADataset().get_data(0)
# X_train, y_train = train_data
# X_test, y_test = test_data

# base_model = LR()
# base_model.train(X_train.values, y_train.values)

# recourse_needed_X_test = recourse_needed(base_model.predict, X_test.values)

# for x0 in ret['x_0']:
#     for i, org in enumerate(recourse_needed_X_test):
#         if np.array_equal(x0, org.round(4)):
#             print("Yippe", i)
#             break

Yippe 24
Yippe 19
Yippe 31
Yippe 0
Yippe 2
Yippe 1


In [ ]:
import os
algo_type = "L1PSD"
model_type = "nn"
dataset_type = "sba"


dir_path = "../results/recourse"
for file in os.listdir(dir_path):
    file_splitted = file.split(sep='_')

    file_path = os.path.join(dir_path, file)
    if os.path.isfile(file_path) and file_splitted[2] == algo_type and \
        file_splitted[0] == model_type  and file_splitted[1] == dataset_type:
        # file_path.endswith('new.pkl'):
        print(file_path)

        seed = int(file_splitted[5].replace(".pkl", ''))

        ret = pd.read_pickle(file_path)
        train_data, test_data = SBADataset().get_data(seed)
        X_train, y_train = train_data
        X_test, y_test = test_data

        base_model = NN(X_train.shape[1])
        base_model.train(X_train.values, y_train.values)

        recourse_needed_X_test = recourse_needed(base_model.predict, X_test.values)
        for x0_i, x0 in enumerate(ret['x_0']):
            for org_i, org in enumerate(recourse_needed_X_test):
                if np.array_equal(x0, org.round(4)):
                    print("Yippe", org_i)
                    ret.loc[x0_i, 'i'] = org_i
                    break
        
        # ret.to_pickle(file_path)
        
        

../results/recourse\nn_sba_L1PSD_0.001_0.1_0.pkl
Yippe 24
Yippe 19
Yippe 31
Yippe 0
Yippe 2
Yippe 1
../results/recourse\nn_sba_L1PSD_0.001_0.1_1.pkl
Yippe 17
Yippe 16
Yippe 27
Yippe 29
Yippe 34
Yippe 4
../results/recourse\nn_sba_L1PSD_0.001_0.1_2.pkl
Yippe 9
Yippe 4
Yippe 30
Yippe 29
Yippe 3
Yippe 17
../results/recourse\nn_sba_L1PSD_0.001_0.1_3.pkl
Yippe 2
Yippe 6
Yippe 27
Yippe 30
Yippe 21
Yippe 26
../results/recourse\nn_sba_L1PSD_0.001_0.1_4.pkl
Yippe 26
Yippe 33
Yippe 34
Yippe 35
Yippe 2
Yippe 36
../results/recourse\nn_sba_L1PSD_0.01_0.1_0.pkl
Yippe 24
Yippe 19
Yippe 31
Yippe 0
Yippe 2
Yippe 1
../results/recourse\nn_sba_L1PSD_0.01_0.1_1.pkl
Yippe 17
Yippe 16
Yippe 27
Yippe 29
Yippe 34
Yippe 4
../results/recourse\nn_sba_L1PSD_0.01_0.1_2.pkl
Yippe 9
Yippe 4
Yippe 30
Yippe 29
Yippe 3
Yippe 17
../results/recourse\nn_sba_L1PSD_0.01_0.1_3.pkl
Yippe 2
Yippe 6
Yippe 27
Yippe 30
Yippe 21
Yippe 26
../results/recourse\nn_sba_L1PSD_0.01_0.1_4.pkl
Yippe 26
Yippe 33
Yippe 34
Yippe 35
Yippe 2
Yipp

In [ ]:
# Combing L1PSD_new with L1PSD for lr_sba dataset

# import os
# algo_type = "L1PSD"
# model_type = "lr"
# dataset_type = "sba"


# dir_path = "../results/recourse"
# for file in os.listdir(dir_path):
#     file_splitted = file.split(sep='_')

#     file_path = os.path.join(dir_path, file)
#     if os.path.isfile(file_path) and file_splitted[2] == algo_type and \
#         file_splitted[0] == model_type  and file_splitted[1] == dataset_type and\
#         file_path.endswith('new.pkl'):
        
#         file_path_old = file_path.replace('_new', '')
#         retOld = pd.read_pickle(file_path_old)
#         retNew = pd.read_pickle(file_path)
#         retFinal = pd.concat((retOld, retNew), axis=0)

#         # retFinal.to_pickle(file_path_old)       

In [466]:
pd.concat((retOld, retNew), axis=0).sort_values(['i'], ignore_index=True)

,algorithm,seed,alpha,lambda,i,x_0,x_r,theta_0
0,L1PSD,4,0.1,2.1,2,"[-0.3064, 0.8538, 0.3964, 0.3547, -1.4489, -0....","[-0.3064, 0.8538, 0.3965, 0.3547, -1.4489, -0....","[0.3122, -0.0656, -0.0824, 0.0148, 0.3491, 0.1..."
1,L1PSD,4,0.1,2.1,26,"[-0.7291, -0.8273, 0.4737, 0.6118, -0.8606, -0...","[-0.7276, -0.8286, 0.4759, 0.6135, -0.8595, -0...","[0.3122, -0.0656, -0.0824, 0.0148, 0.3491, 0.1..."
2,L1PSD,4,0.1,2.1,33,"[0.7876, -1.012, -0.597, -0.6737, -1.2794, -0....","[0.7886, -1.013, -0.5952, -0.672, -1.2811, -0....","[0.3122, -0.0656, -0.0824, 0.0148, 0.3491, 0.1..."
3,L1PSD,4,0.1,2.1,34,"[-0.044, -1.012, 0.5769, 0.6118, -1.1597, -0.2...","[-0.0443, -1.0114, 0.5748, 0.6099, -1.1601, -0...","[0.3122, -0.0656, -0.0824, 0.0148, 0.3491, 0.1..."
4,L1PSD,4,0.1,2.1,35,"[1.7451, 0.8538, -0.5633, -0.6737, -0.5615, -0...","[1.7451, 0.8538, -0.5633, -0.6737, -0.5615, -0...","[0.3122, -0.0656, -0.0824, 0.0148, 0.3491, 0.1..."
5,L1PSD,4,0.1,2.1,36,"[-0.049, 1.0001, 0.8535, 0.8689, -0.382, -0.22...","[-0.049, 1.0001, 0.8535, 0.8689, -0.382, -0.22...","[0.3122, -0.0656, -0.0824, 0.0148, 0.3491, 0.1..."


### Comparison of J values of L1 and Linf

In [16]:
num_i = 27

file_path = "../results/recourse/lr_sba_ROARL1_1.0_0.1_1.pkl"
ret = pd.read_pickle(file_path)
# np.sum(np.abs((ret['x_r'] - ret['x_0'])[num_i]))
ret

,algorithm,seed,alpha,lambda,i,x_0,x_r,theta_0
0,ROARL1,1,0.1,1.0,0,"[0.3262, 0.98, 0.7742, 0.8689, -1.4489, -0.249...","[0.3263, 0.9796, 0.7743, 0.8693, -1.4491, -0.2...","[0.3199, -0.1559, -0.035, 0.1301, 0.3266, 0.11..."
1,ROARL1,1,0.1,1.0,1,"[-1.3662, -0.8273, 0.4021, 0.3547, -0.8805, -0...","[-1.3662, -0.8277, 0.402, 0.3551, -0.8803, -0....","[0.3199, -0.1559, -0.035, 0.1301, 0.3266, 0.11..."
2,ROARL1,1,0.1,1.0,2,"[0.7914, -0.8108, 0.579, 0.6118, -1.0401, -0.1...","[0.7915, -0.8109, 0.579, 0.6115, -1.0399, -0.1...","[0.3199, -0.1559, -0.035, 0.1301, 0.3266, 0.11..."
3,ROARL1,1,0.1,1.0,3,"[-0.5048, 0.8172, 0.4709, 0.6118, -0.9803, -0....","[-0.5047, 0.8172, 0.4708, 0.6119, -0.98, -0.15...","[0.3199, -0.1559, -0.035, 0.1301, 0.3266, 0.11..."
4,ROARL1,1,0.1,1.0,4,"[1.6246, -1.012, 0.4898, 0.6118, -0.9503, -0.2...","[1.6246, -1.0119, 0.4898, 0.6119, -0.9502, -0....","[0.3199, -0.1559, -0.035, 0.1301, 0.3266, 0.11..."
5,ROARL1,1,0.1,1.0,5,"[1.6545, -1.012, 0.9075, 0.8689, -0.9404, -0.2...","[1.6546, -1.012, 0.9077, 0.869, -0.9405, -0.22...","[0.3199, -0.1559, -0.035, 0.1301, 0.3266, 0.11..."
6,ROARL1,1,0.1,1.0,6,"[-1.2043, -1.012, 0.8472, 0.8689, -1.1897, 0.0...","[-1.2038, -1.012, 0.8471, 0.8689, -1.1896, 0.0...","[0.3199, -0.1559, -0.035, 0.1301, 0.3266, 0.11..."
7,ROARL1,1,0.1,1.0,7,"[0.6726, -1.012, 0.5874, 0.6118, 0.665, -0.226...","[0.6728, -1.012, 0.5872, 0.6117, 0.6646, -0.22...","[0.3199, -0.1559, -0.035, 0.1301, 0.3266, 0.11..."
8,ROARL1,1,0.1,1.0,8,"[0.5124, 0.8538, -1.1818, -1.1879, -1.1697, -0...","[0.5125, 0.8543, -1.1811, -1.1885, -1.1696, -0...","[0.3199, -0.1559, -0.035, 0.1301, 0.3266, 0.11..."
9,ROARL1,1,0.1,1.0,9,"[-0.8181, -0.6828, 0.8345, 0.8689, -1.1797, -0...","[-0.8176, -0.6831, 0.8344, 0.8692, -1.1796, -0...","[0.3199, -0.1559, -0.035, 0.1301, 0.3266, 0.11..."


In [17]:
num_i1 = 2

file_path = "../results/recourse/lr_sba_L1PSD_2.8_0.1_1.pkl"
ret1 = pd.read_pickle(file_path)
# np.sum(np.abs((ret1['x_r'] - ret1['x_0'])[num_i]))
ret1

,algorithm,seed,alpha,lambda,i,x_0,x_r,theta_0
0,L1PSD,1,0.1,2.8,17,"[-1.117, 0.98, 0.7096, 0.6118, -0.8706, -0.249...","[-1.1191, 0.9797, 0.7071, 0.6093, -0.8726, -0....","[0.3199, -0.1559, -0.035, 0.1301, 0.3266, 0.11..."
1,L1PSD,1,0.1,2.8,16,"[-1.1302, 0.8538, 0.6794, 0.6118, -0.8207, -0....","[-1.1322, 0.855, 0.677, 0.6097, -0.8222, -0.20...","[0.3199, -0.1559, -0.035, 0.1301, 0.3266, 0.11..."
2,L1PSD,1,0.1,2.8,27,"[-1.0711, 0.8538, 0.0819, 0.0976, -0.9503, -0....","[-1.07, 0.8559, 0.0809, 0.095, -0.9494, -0.249...","[0.3199, -0.1559, -0.035, 0.1301, 0.3266, 0.11..."
3,L1PSD,1,0.1,2.8,29,"[-0.2114, -0.6828, 0.7777, 0.8689, -0.9304, -0...","[-0.2134, -0.6829, 0.7753, 0.8665, -0.9324, -0...","[0.3199, -0.1559, -0.035, 0.1301, 0.3266, 0.11..."
4,L1PSD,1,0.1,2.8,34,"[1.2417, -1.012, 0.8079, 0.8689, -1.2993, -0.2...","[1.2425, -1.0092, 0.8067, 0.872, -1.2977, -0.2...","[0.3199, -0.1559, -0.035, 0.1301, 0.3266, 0.11..."
5,L1PSD,1,0.1,2.8,4,"[1.6246, -1.012, 0.4898, 0.6118, -0.9503, -0.2...","[1.6239, -1.0117, 0.4874, 0.6149, -0.9523, -0....","[0.3199, -0.1559, -0.035, 0.1301, 0.3266, 0.11..."


In [85]:
J = RecourseCost(ret['x_0'][num_i], ret['lambda'][num_i])
thetaP = calThetaAdv_linf(ret['x_r'][num_i], ret['theta_0'][num_i][:-1], 
                                  ret['theta_0'][num_i][[-1]], ret['alpha'][num_i])
print(J.eval(ret['x_r'][num_i], thetaP[:-1], thetaP[[-1]], breakdown=True))

(array([0.00026885]), np.float64(4.367), array([0.00463585]))


In [86]:
J = RecourseCost(ret1['x_0'][num_i1], ret1['lambda'][num_i1])
thetaP = calThetaAdv_l1(ret1['x_r'][num_i1], ret1['theta_0'][num_i1], ret1['alpha'][num_i1])
print(J.eval(ret1['x_r'][num_i1], thetaP[:-1], thetaP[[-1]], breakdown=True))

(array([4.93059402e-05]), np.float64(4.9428), array([0.00499211]))


In [111]:
ret['x_r'][0].shape

(25,)